# 第14章（二）：电路交换与分组交换
# Chapter 14 Part 2: Circuit Switching vs Packet Switching

**Cambridge A-Level Computer Science (9618)**

---

## 本节学习目标 (Learning Objectives)

| 编号 | 目标 | Keywords |
|------|------|----------|
| 1 | 理解电路交换的原理和过程 | Circuit Switching, Dedicated Path |
| 2 | 理解分组交换的原理和过程 | Packet Switching, Packet, Router |
| 3 | 比较两种交换方式的优缺点 | Comparison, Trade-offs |
| 4 | 理解数据包的结构 | Header, Payload, Trailer |
| 5 | 理解路由器在分组交换中的作用 | Router, Routing Table |
| 6 | 解释为什么互联网选择了分组交换 | Internet, Resilience |

---
## 1. 电路交换 (Circuit Switching)

### 1.1 什么是电路交换？

> **电路交换 (Circuit Switching)** 是一种通信方式，在通信开始前，先在发送方和接收方之间建立一条**专用的物理通路 (dedicated path)**，通信期间这条通路被独占使用，通信结束后释放。

### 1.2 现实类比：打电话

想想你用座机打电话的过程：
1. 你拨号 → 电话交换机为你和对方建立一条专线
2. 通话中 → 这条线路只属于你们两个人
3. 挂断 → 线路释放，别人可以使用

即使你们沉默了30秒没说话，这条线路依然被占用！这就是电路交换的核心特征。

### 1.3 电路交换的三个阶段

| 阶段 | 名称 | 说明 |
|------|------|------|
| 1 | **建立连接 (Circuit Establishment)** | 发送方发出请求，沿路的交换机逐个分配资源，直到建立完整通路 |
| 2 | **数据传输 (Data Transfer)** | 数据沿着固定路径传输，全程路径不变 |
| 3 | **释放连接 (Circuit Disconnect)** | 通信结束，沿路释放所有资源 |

### 1.4 电路交换的优点
- **稳定的带宽**：专用通路保证了固定的传输速率
- **低延迟**：数据不需要存储转发，路径已经建立好
- **数据按序到达**：只有一条路径，不会乱序
- **适合实时通信**：如传统电话通话

### 1.5 电路交换的缺点
- **资源浪费**：即使没有数据传输（如通话中的沉默），线路也被占用
- **建立连接需要时间**：传输前必须等待连接建立
- **缺乏灵活性**：如果路径中某个节点故障，整个连接断开
- **不适合突发数据**：互联网流量是突发性的（burst），电路交换效率低

In [ ]:
# === 可视化：电路交换的三个阶段 ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('电路交换的三个阶段 (Three Phases of Circuit Switching)', fontsize=18, fontweight='bold')

# 网络拓扑节点
nodes = {
    'A': (1, 5), 'S1': (3, 7), 'S2': (5, 5), 'S3': (3, 3),
    'S4': (5, 7), 'S5': (7, 5), 'B': (9, 5),
}
# 所有可能的连线
all_links = [('A','S1'), ('A','S3'), ('S1','S2'), ('S1','S4'), ('S2','S3'),
             ('S2','S5'), ('S4','S5'), ('S5','B'), ('S3','S5')]
# 选定的路径
path = [('A','S1'), ('S1','S4'), ('S4','S5'), ('S5','B')]

titles = [
    '阶段1: 建立连接\n(Circuit Establishment)',
    '阶段2: 数据传输\n(Data Transfer)',
    '阶段3: 释放连接\n(Circuit Disconnect)',
]
stage_colors = ['#F39C12', '#27AE60', '#E74C3C']

for idx, (ax, title, stage_color) in enumerate(zip(axes, titles, stage_colors)):
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 9)
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', color=stage_color)
    
    # 画所有连线（灰色）
    for n1, n2 in all_links:
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        ax.plot([x1, x2], [y1, y2], '-', color='#DDD', linewidth=2, zorder=1)
    
    # 画选定路径
    if idx == 0:  # 建立连接 - 逐步高亮
        for i, (n1, n2) in enumerate(path):
            x1, y1 = nodes[n1]
            x2, y2 = nodes[n2]
            alpha = 0.3 + 0.2 * i
            ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                        arrowprops=dict(arrowstyle='->', color=stage_color, lw=3, alpha=alpha))
            ax.text((x1+x2)/2, (y1+y2)/2 + 0.4, f'({i+1})', fontsize=8, ha='center',
                    color=stage_color, fontweight='bold')
    elif idx == 1:  # 数据传输 - 全部高亮
        for n1, n2 in path:
            x1, y1 = nodes[n1]
            x2, y2 = nodes[n2]
            ax.plot([x1, x2], [y1, y2], '-', color=stage_color, linewidth=4, zorder=2)
        # 数据流动箭头
        data_positions = [(1.5, 5.5), (3.5, 6.8), (5.5, 6.5), (7.5, 5.2)]
        for dx, dy in data_positions:
            ax.annotate('', xy=(dx+0.8, dy-0.1), xytext=(dx, dy),
                        arrowprops=dict(arrowstyle='->', color='blue', lw=2))
        ax.text(5, 1.5, '专用通路，数据持续传输', ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))
    else:  # 释放连接 - 虚线
        for n1, n2 in path:
            x1, y1 = nodes[n1]
            x2, y2 = nodes[n2]
            ax.plot([x1, x2], [y1, y2], '--', color=stage_color, linewidth=2, zorder=2, alpha=0.5)
        ax.text(5, 1.5, '释放资源，线路可供他人使用', ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='#FADBD8', edgecolor='#E74C3C'))
    
    # 画节点
    for name, (x, y) in nodes.items():
        if name in ['A', 'B']:
            color = '#3498DB' if name == 'A' else '#E74C3C'
            ax.add_patch(plt.Circle((x, y), 0.5, facecolor=color, edgecolor='black', linewidth=2, zorder=3))
            label = '发送方 A' if name == 'A' else '接收方 B'
            ax.text(x, y, label, ha='center', va='center', fontsize=7, color='white', fontweight='bold', zorder=4)
        else:
            ax.add_patch(plt.Circle((x, y), 0.4, facecolor='#F5F5F5', edgecolor='black', linewidth=1.5, zorder=3))
            ax.text(x, y, name, ha='center', va='center', fontsize=9, zorder=4)

plt.tight_layout()
plt.savefig('02_circuit_switching.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 2. 分组交换 (Packet Switching)

### 2.1 什么是分组交换？

> **分组交换 (Packet Switching)** 是一种通信方式，数据在发送前被分割成多个小的**数据包 (packets)**，每个数据包独立地通过网络传输，可能经过不同的路径，到达目的地后重新组装。

### 2.2 现实类比：快递包裹

你要搬家，有10箱东西要从北京寄到上海：
- 每个箱子上贴了标签（发件地址、收件地址、第几箱/共几箱）
- 每个箱子由不同的快递员通过不同路线运送
- 有的走高速公路，有的走铁路，有的走航空
- 到达顺序可能不一样：第5箱可能比第1箱先到
- 到达后按编号重新排列组装

### 2.3 数据包的结构 (Packet Structure)

每个数据包由三部分组成：

```
┌──────────────┬──────────────────────────────┬──────────────┐
│   报头        │         载荷                  │    报尾       │
│   Header     │         Payload               │    Trailer   │
├──────────────┼──────────────────────────────┼──────────────┤
│ - 源IP地址    │                              │ - 校验和      │
│ - 目标IP地址  │      实际传输的数据            │   (Checksum) │
│ - 序列号      │                              │              │
│ - TTL        │                              │              │
│ - 协议类型    │                              │              │
└──────────────┴──────────────────────────────┴──────────────┘
```

**报头 (Header) 的关键字段**：
- **源IP地址 (Source IP)**：数据从哪里来
- **目标IP地址 (Destination IP)**：数据要到哪里去
- **序列号 (Sequence Number)**：这是第几个包，用于重组
- **TTL (Time to Live)**：数据包最多经过多少个路由器（防止无限循环）
- **协议类型**：载荷使用的上层协议（TCP/UDP）

### 2.4 为什么要把数据分成包？

1. **公平共享网络**：大文件不会独占整条线路，其他用户的小数据包也能穿插传输
2. **容错性**：如果一个包丢失了，只需要重传那一个包，不需要重传整个文件
3. **灵活路由**：每个包可以走最优的路径，避开拥堵

In [ ]:
# === 可视化：数据包的结构 ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('数据包结构详解 (Packet Structure)', fontsize=18, fontweight='bold', pad=20)

# 主体数据包
y_main = 7
# Header
ax.add_patch(mpatches.FancyBboxPatch((1, y_main-0.8), 4, 1.6, boxstyle='round,pad=0.05',
                                      facecolor='#3498DB', edgecolor='black', linewidth=2, alpha=0.85))
ax.text(3, y_main, 'Header (报头)', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
# Payload
ax.add_patch(mpatches.FancyBboxPatch((5.2, y_main-0.8), 6.5, 1.6, boxstyle='round,pad=0.05',
                                      facecolor='#2ECC71', edgecolor='black', linewidth=2, alpha=0.85))
ax.text(8.45, y_main, 'Payload (载荷/有效数据)', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
# Trailer
ax.add_patch(mpatches.FancyBboxPatch((11.9, y_main-0.8), 2.5, 1.6, boxstyle='round,pad=0.05',
                                      facecolor='#E74C3C', edgecolor='black', linewidth=2, alpha=0.85))
ax.text(13.15, y_main, 'Trailer\n(报尾)', ha='center', va='center', fontsize=12, fontweight='bold', color='white')

# Header 详细字段
header_fields = [
    ('源IP地址\nSource IP', '#AED6F1'),
    ('目标IP地址\nDest IP', '#AED6F1'),
    ('序列号\nSeq No.', '#D4E6F1'),
    ('TTL', '#D4E6F1'),
    ('协议\nProtocol', '#D4E6F1'),
    ('头长度\nHdr Len', '#E8F6F3'),
]

x_start = 0.5
for name, color in header_fields:
    w = 2.4
    ax.add_patch(mpatches.FancyBboxPatch((x_start, 3), w, 1.5, boxstyle='round,pad=0.05',
                                          facecolor=color, edgecolor='#3498DB', linewidth=1.5))
    ax.text(x_start + w/2, 3.75, name, ha='center', va='center', fontsize=8, fontweight='bold')
    x_start += w + 0.15

# 从Header指向详细字段
ax.annotate('', xy=(7, 4.5), xytext=(3, y_main - 0.8),
            arrowprops=dict(arrowstyle='->', color='#3498DB', lw=2,
                           connectionstyle='arc3,rad=0.2'))
ax.text(0.5, 5, 'Header包含的字段:', fontsize=11, fontweight='bold', color='#3498DB')

# Trailer 详细
ax.add_patch(mpatches.FancyBboxPatch((12, 3), 2.5, 1.5, boxstyle='round,pad=0.05',
                                      facecolor='#FADBD8', edgecolor='#E74C3C', linewidth=1.5))
ax.text(13.25, 3.75, '校验和\nChecksum\n(错误检测)', ha='center', va='center', fontsize=9, fontweight='bold')
ax.annotate('', xy=(13.15, 4.5), xytext=(13.15, y_main - 0.8),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2))

# 关键说明
notes = [
    '序列号(Sequence Number): 用于在目的地重新组装数据包的正确顺序',
    'TTL(Time to Live): 每经过一个路由器减1，到0就丢弃（防止无限循环）',
    '校验和(Checksum): 接收方用它来检测传输过程中是否发生了错误',
]
for i, note in enumerate(notes):
    ax.text(0.5, 1.8 - i * 0.7, f'  {note}', fontsize=9, color='#333',
            style='italic')

plt.tight_layout()
plt.savefig('02_packet_structure.png', dpi=150, bbox_inches='tight')


In [ ]:
# === 可视化：分组交换 — 数据包走不同路径 ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('分组交换：数据包走不同路径到达目的地\n(Packet Switching: Packets Take Different Routes)',
             fontsize=16, fontweight='bold', pad=20)

# 网络节点
nodes = {
    'Src': (1, 6),
    'R1': (4, 9), 'R2': (4, 6), 'R3': (4, 3),
    'R4': (8, 9), 'R5': (8, 6), 'R6': (8, 3),
    'R7': (11, 8), 'R8': (11, 4),
    'Dst': (14, 6),
}

# 所有连线
all_edges = [
    ('Src','R1'), ('Src','R2'), ('Src','R3'),
    ('R1','R2'), ('R2','R3'), ('R1','R4'), ('R2','R5'), ('R3','R6'),
    ('R4','R5'), ('R5','R6'), ('R4','R7'), ('R5','R7'), ('R5','R8'), ('R6','R8'),
    ('R7','Dst'), ('R8','Dst'),
]

# 画所有连线（灰色背景）
for n1, n2 in all_edges:
    x1, y1 = nodes[n1]
    x2, y2 = nodes[n2]
    ax.plot([x1, x2], [y1, y2], '-', color='#DDD', linewidth=2, zorder=1)

# 三个数据包的路径（用不同颜色标注）
packet_paths = [
    {
        'path': [('Src','R1'), ('R1','R4'), ('R4','R7'), ('R7','Dst')],
        'color': '#E74C3C', 'label': 'Packet 1 (上方路径)',
    },
    {
        'path': [('Src','R2'), ('R2','R5'), ('R5','R7'), ('R7','Dst')],
        'color': '#27AE60', 'label': 'Packet 2 (中间路径)',
    },
    {
        'path': [('Src','R3'), ('R3','R6'), ('R6','R8'), ('R8','Dst')],
        'color': '#3498DB', 'label': 'Packet 3 (下方路径)',
    },
]

for pkt in packet_paths:
    for n1, n2 in pkt['path']:
        x1, y1 = nodes[n1]
        x2, y2 = nodes[n2]
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='->', color=pkt['color'], lw=3, alpha=0.7))

# 画节点
for name, (x, y) in nodes.items():
    if name in ['Src', 'Dst']:
        color = '#2C3E50'
        radius = 0.6
        label = '发送方\nSource' if name == 'Src' else '接收方\nDest'
        fc = 'white'
    else:
        color = '#95A5A6'
        radius = 0.45
        label = name
        fc = 'white'
    ax.add_patch(plt.Circle((x, y), radius, facecolor='lightyellow', edgecolor=color,
                             linewidth=2, zorder=3))
    ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold',
            color=color, zorder=4)

# 图例
for i, pkt in enumerate(packet_paths):
    ax.text(1, 1.5 - i * 0.6, f'  {pkt["label"]}', fontsize=11,
            color=pkt['color'], fontweight='bold')
    ax.plot([0.5, 0.9], [1.5 - i * 0.6, 1.5 - i * 0.6], '-',
            color=pkt['color'], linewidth=3)

# 关键概念框
ax.text(10, 1.5, '关键概念：\n每个数据包可以走不同路径\n到达后按序列号重组',
        fontsize=11, ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#FEF9E7', edgecolor='#F39C12', linewidth=2))

plt.tight_layout()
plt.savefig('02_packet_switching_routes.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 3. 路由器在分组交换中的作用 (Role of Routers)

### 3.1 路由器做什么？

路由器 (Router) 是分组交换网络中的核心设备。每个路由器做以下工作：

1. **接收数据包** — 从一个接口收到数据包
2. **检查目标IP** — 读取数据包头部的目标IP地址
3. **查路由表 (Routing Table)** — 找到通往目标的最佳下一跳 (Next Hop)
4. **转发数据包** — 把数据包从对应的接口发出去
5. **TTL减1** — 如果TTL变成0，丢弃该包（防止死循环）

### 3.2 路由表 (Routing Table)

每个路由器维护一张路由表，记录：
- 目标网络地址
- 下一跳路由器的地址
- 使用的接口
- 路径的"代价"（跳数、延迟、带宽等）

### 3.3 为什么不同数据包走不同路径？

因为每个路由器在转发每个数据包时，都会**独立做决策**：
- 如果某条路径拥堵了，路由器会选择另一条路径
- 如果某条路径断了，路由器会绕过故障点
- 路由表会动态更新，反映网络的实时状态

In [ ]:
# === 交互式演示：模拟路由器转发数据包 ===

class Router:
    '''模拟一个简单的路由器'''
    
    def __init__(self, name):
        self.name = name
        self.routing_table = {}  # {目标网络: (下一跳, 接口, 代价)}
    
    def add_route(self, dest_network, next_hop, interface, cost):
        self.routing_table[dest_network] = (next_hop, interface, cost)
    
    def forward(self, packet):
        dest_ip = packet['dest_ip']
        # 查找匹配的路由
        dest_network = '.'.join(dest_ip.split('.')[:3]) + '.0'
        
        if dest_network in self.routing_table:
            next_hop, interface, cost = self.routing_table[dest_network]
            # TTL减1
            packet['ttl'] -= 1
            if packet['ttl'] <= 0:
                print(f'  [{self.name}] TTL expired! Packet dropped.')
                return None
            print(f'  [{self.name}] 收到数据包 -> 目标: {dest_ip}')
            print(f'           查路由表: {dest_network} -> 下一跳: {next_hop} (接口: {interface}, 代价: {cost})')
            print(f'           TTL: {packet["ttl"]+1} -> {packet["ttl"]}')
            return next_hop
        else:
            print(f'  [{self.name}] 没有匹配的路由! 丢弃数据包。')
            return None

class PacketSwitchingSimulation:
    '''模拟分组交换的数据包转发过程'''
    
    def __init__(self):
        self.routers = {}
        self._setup_network()
    
    def _setup_network(self):
        # 创建路由器
        r1 = Router('R1-Beijing')
        r2 = Router('R2-Shanghai')
        r3 = Router('R3-Guangzhou')
        r4 = Router('R4-Shenzhen')
        
        # 配置路由表
        r1.add_route('10.0.4.0', 'R2-Shanghai', 'eth1', 2)
        r1.add_route('10.0.3.0', 'R3-Guangzhou', 'eth2', 3)
        r2.add_route('10.0.4.0', 'R4-Shenzhen', 'eth1', 1)
        r3.add_route('10.0.4.0', 'R4-Shenzhen', 'eth1', 1)
        r4.add_route('10.0.4.0', 'DESTINATION', 'eth0', 0)
        
        self.routers = {'R1-Beijing': r1, 'R2-Shanghai': r2,
                         'R3-Guangzhou': r3, 'R4-Shenzhen': r4}
    
    def send_packet(self, src_ip, dest_ip, data, seq_num):
        packet = {
            'src_ip': src_ip,
            'dest_ip': dest_ip,
            'data': data,
            'seq_num': seq_num,
            'ttl': 10,
        }
        
        print(f'\n--- 数据包 #{seq_num} ---')
        print(f'  源IP: {src_ip} -> 目标IP: {dest_ip}')
        print(f'  数据: "{data}"')
        print(f'  TTL: {packet["ttl"]}')
        print()
        
        current_router = 'R1-Beijing'  # 起始路由器
        hops = 0
        while current_router and current_router != 'DESTINATION':
            if current_router in self.routers:
                current_router = self.routers[current_router].forward(packet)
                hops += 1
            else:
                break
        
        if current_router == 'DESTINATION':
            print(f'  数据包 #{seq_num} 到达目的地! 经过 {hops} 跳。')
        print()

# 运行模拟
sim = PacketSwitchingSimulation()
print('=' * 60)
print('分组交换模拟：从北京发送数据到深圳')
print('原始数据: "Hello from Beijing!"')
print('数据被分成3个包')
print('=' * 60)

# 模拟3个数据包
sim.send_packet('10.0.1.100', '10.0.4.50', 'Hello ', 1)
sim.send_packet('10.0.1.100', '10.0.4.50', 'from ', 2)
sim.send_packet('10.0.1.100', '10.0.4.50', 'Beijing!', 3)

print('=' * 60)
print('所有数据包到达后，按序列号重组: "Hello from Beijing!"')


---
## 4. 电路交换 vs 分组交换：详细对比

这是考试的**超高频考点**！必须能够清晰地对比两者。

| 特征 | 电路交换 (Circuit Switching) | 分组交换 (Packet Switching) |
|------|-----|-----|
| **连接方式** | 建立专用物理通路 | 无需建立连接，每个包独立路由 |
| **路径** | 固定路径，全程不变 | 每个数据包可能走不同路径 |
| **资源利用** | 独占资源，即使空闲也不释放 | 共享资源，按需使用 |
| **带宽** | 保证固定带宽 | 带宽动态分配 |
| **延迟** | 建立连接后延迟低且恒定 | 延迟可能变化（排队等待） |
| **数据顺序** | 数据按序到达 | 数据包可能乱序到达，需重组 |
| **容错性** | 路径中任一点故障 = 连接断开 | 路由器故障后可绕道，更鲁棒 |
| **适合场景** | 实时通信（语音电话） | 数据通信（互联网） |
| **效率** | 低（沉默期间浪费带宽） | 高（多用户共享同一网络） |
| **典型应用** | 传统电话网络(PSTN) | 互联网 |

In [ ]:
# === 可视化：电路交换 vs 分组交换 并排对比 ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle('电路交换 vs 分组交换', fontsize=20, fontweight='bold')

# === 左图：电路交换 ===
ax1.set_xlim(0, 10); ax1.set_ylim(0, 10); ax1.axis('off')
ax1.set_title('电路交换 (Circuit Switching)', fontsize=15, fontweight='bold', color='#E74C3C')

# 时间轴
ax1.text(0.5, 9.5, '时间 →', fontsize=10, color='#666')
for t in range(10):
    ax1.axvline(x=t, color='#EEE', linewidth=0.5)

# 三条线路
lines = [('线路 A→B', 8, '#E74C3C'), ('线路 C→D', 6, '#3498DB'), ('线路 E→F', 4, '#27AE60')]
for name, y, color in lines:
    ax1.text(0.2, y, name, fontsize=9, va='center', fontweight='bold', color=color)

# A→B: 占满整条线
ax1.add_patch(mpatches.FancyBboxPatch((1.5, 7.5), 6.5, 0.8, boxstyle='round',
                                       facecolor='#E74C3C', alpha=0.6))
ax1.text(4.75, 7.9, '完整通话（含沉默期间也占用）', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

# C→D: 占满整条线
ax1.add_patch(mpatches.FancyBboxPatch((2.5, 5.5), 5, 0.8, boxstyle='round',
                                       facecolor='#3498DB', alpha=0.6))
ax1.text(5, 5.9, '通话期间独占', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

# E→F: 等待
ax1.add_patch(mpatches.Rectangle((1, 3.6), 1, 0.6, facecolor='#FF6B6B', alpha=0.5))
ax1.text(1.5, 3.9, '等待', ha='center', va='center', fontsize=7, color='white')
ax1.add_patch(mpatches.FancyBboxPatch((3, 3.5), 4, 0.8, boxstyle='round',
                                       facecolor='#27AE60', alpha=0.6))
ax1.text(5, 3.9, '线路可用后才能通话', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax1.text(5, 2, '问题：资源被独占\n沉默期间带宽浪费', ha='center', va='center', fontsize=11,
         color='#E74C3C', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#FDEBD0', edgecolor='#E74C3C'))

# === 右图：分组交换 ===
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.set_title('分组交换 (Packet Switching)', fontsize=15, fontweight='bold', color='#27AE60')

ax2.text(0.5, 9.5, '时间 →', fontsize=10, color='#666')
for t in range(10):
    ax2.axvline(x=t, color='#EEE', linewidth=0.5)

# 共享线路
ax2.text(0.2, 8, '共享\n链路', fontsize=9, va='center', fontweight='bold', color='#333')

# 不同用户的数据包交替使用线路
packets = [
    (1, 7.5, 0.8, '#E74C3C', 'A1'),
    (2, 7.5, 0.8, '#3498DB', 'C1'),
    (3, 7.5, 0.8, '#27AE60', 'E1'),
    (4, 7.5, 0.8, '#E74C3C', 'A2'),
    (5, 7.5, 0.8, '#3498DB', 'C2'),
    (6, 7.5, 0.8, '#27AE60', 'E2'),
    (7, 7.5, 0.8, '#E74C3C', 'A3'),
    (8, 7.5, 0.8, '#3498DB', 'C3'),
]

for x, y, w, color, label in packets:
    ax2.add_patch(mpatches.FancyBboxPatch((x, y), w, 0.8, boxstyle='round',
                                           facecolor=color, alpha=0.7))
    ax2.text(x + w/2, y + 0.4, label, ha='center', va='center',
             fontsize=8, color='white', fontweight='bold')

# 图例
legend_items = [('A的数据包', '#E74C3C'), ('C的数据包', '#3498DB'), ('E的数据包', '#27AE60')]
for i, (name, color) in enumerate(legend_items):
    ax2.add_patch(mpatches.Rectangle((1 + i*3, 6), 0.8, 0.4, facecolor=color, alpha=0.7))
    ax2.text(2 + i*3, 6.2, name, fontsize=9, va='center', color=color, fontweight='bold')

ax2.text(5, 5, '所有用户的数据包交替\n使用同一条链路', ha='center', va='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))

ax2.text(5, 2, '优势：资源共享\n带宽充分利用，无浪费', ha='center', va='center', fontsize=11,
         color='#27AE60', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#D5F5E3', edgecolor='#27AE60'))

plt.tight_layout()
plt.savefig('02_comparison.png', dpi=150, bbox_inches='tight')


In [ ]:
# === 可视化：带宽利用率对比 ===
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('带宽利用率对比 (Bandwidth Utilisation Comparison)', fontsize=16, fontweight='bold')

t = np.linspace(0, 10, 1000)

# 上图：电路交换的带宽使用
ax1.set_title('电路交换：即使没有数据传输，带宽也被占用', fontsize=13, color='#E74C3C')
# 实际数据传输（突发性的）
np.random.seed(42)
actual_data = np.zeros_like(t)
for _ in range(15):
    center = np.random.uniform(0.5, 9.5)
    width = np.random.uniform(0.1, 0.4)
    height = np.random.uniform(50, 100)
    actual_data += height * np.exp(-((t - center) / width) ** 2)
actual_data = np.clip(actual_data, 0, 100)

# 分配的带宽（固定）
ax1.fill_between(t, 0, 100, alpha=0.2, color='#E74C3C', label='分配的带宽(全程占用)')
ax1.fill_between(t, 0, actual_data, alpha=0.8, color='#E74C3C', label='实际使用的带宽')
ax1.fill_between(t, actual_data, 100, alpha=0.3, color='yellow', label='浪费的带宽')
ax1.set_ylabel('带宽 %')
ax1.set_ylim(0, 110)
ax1.legend(loc='upper right')

used = np.mean(actual_data)
ax1.text(5, 105, f'平均利用率: {used:.0f}% (其余 {100-used:.0f}% 被浪费)',
         ha='center', fontsize=11, fontweight='bold', color='#E74C3C')

# 下图：分组交换的带宽使用
ax2.set_title('分组交换：多用户共享带宽，利用率高', fontsize=13, color='#27AE60')

colors = ['#E74C3C', '#3498DB', '#27AE60', '#F39C12', '#9B59B6']
labels = ['用户A', '用户B', '用户C', '用户D', '用户E']
bottom = np.zeros_like(t)

for i, (color, label) in enumerate(zip(colors, labels)):
    np.random.seed(i * 10 + 1)
    user_data = np.zeros_like(t)
    for _ in range(8):
        center = np.random.uniform(0.5, 9.5)
        width = np.random.uniform(0.1, 0.3)
        height = np.random.uniform(10, 30)
        user_data += height * np.exp(-((t - center) / width) ** 2)
    user_data = np.clip(user_data, 0, 30)
    ax2.fill_between(t, bottom, bottom + user_data, alpha=0.7, color=color, label=label)
    bottom += user_data

total = np.clip(bottom, 0, 100)
ax2.set_ylabel('带宽 %')
ax2.set_xlabel('时间')
ax2.set_ylim(0, 110)
ax2.legend(loc='upper right', ncol=5)

avg_total = np.mean(np.clip(bottom, 0, 100))
ax2.text(5, 105, f'平均利用率: {min(avg_total, 95):.0f}% (多用户共享，高效利用)',
         ha='center', fontsize=11, fontweight='bold', color='#27AE60')

plt.tight_layout()
plt.savefig('02_bandwidth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 5. 为什么互联网选择了分组交换？(Deep Dive)

### 5.1 历史背景

1960年代，美国国防部的ARPA想建一个即使部分被摧毁也能继续工作的通信网络。

- **电路交换**：一条线路断了，整个通信就中断了。不行！
- **分组交换**：一条路断了，数据包可以绕道走其他路。完美！

### 5.2 技术原因

1. **互联网流量的突发性 (Bursty Nature)**
   - 你浏览网页：点击→数据涌入→阅读（无数据）→再点击
   - 如果用电路交换，阅读期间带宽全浪费了
   - 分组交换让你阅读时，别人可以使用同一条线路

2. **可靠性 (Reliability)**
   - 互联网没有中心控制点，每个路由器独立决策
   - 某个路由器坏了？其他路由器自动绕道
   - 这就是为什么互联网如此健壮

3. **可扩展性 (Scalability)**
   - 电路交换要为每对通信者维护专用线路，用户一多就爆炸
   - 分组交换让所有用户共享同一个网络，加用户只需加路由器

4. **成本 (Cost)**
   - 共享带宽 = 每个用户的成本更低
   - 想想：一根光纤可以同时为百万用户服务

### 5.3 电路交换并未消失

虽然互联网用分组交换，但电路交换仍然存在：
- 传统电话网络 (PSTN)
- 某些专用的高可靠性网络
- 虚电路 (Virtual Circuit) — 结合了两者的优点

In [ ]:
# === 可视化：容错性对比 — 节点故障时的表现 ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('容错性对比：节点故障时发生什么？', fontsize=18, fontweight='bold')

# 网络节点
nodes = {
    'A': (1, 5), 'N1': (3, 7), 'N2': (5, 5), 'N3': (3, 3),
    'N4': (7, 7), 'N5': (7, 3), 'B': (9, 5),
}
edges = [('A','N1'), ('A','N3'), ('N1','N2'), ('N1','N4'), ('N2','N3'),
         ('N2','N4'), ('N2','N5'), ('N3','N5'), ('N4','B'), ('N5','B')]

# 左图：电路交换 — 节点故障导致连接断开
ax1.set_xlim(0, 10); ax1.set_ylim(0, 9); ax1.axis('off')
ax1.set_title('电路交换：节点N2故障 → 连接中断!', fontsize=14, fontweight='bold', color='#E74C3C')

# 画所有边
for n1, n2 in edges:
    x1, y1 = nodes[n1]; x2, y2 = nodes[n2]
    ax1.plot([x1, x2], [y1, y2], '-', color='#DDD', linewidth=2)

# 原路径高亮 (现在断了)
orig_path = [('A','N1'), ('N1','N2'), ('N2','N4'), ('N4','B')]
for n1, n2 in orig_path:
    x1, y1 = nodes[n1]; x2, y2 = nodes[n2]
    ax1.plot([x1, x2], [y1, y2], '--', color='#E74C3C', linewidth=3, alpha=0.5)

# 画节点
for name, (x, y) in nodes.items():
    if name == 'N2':  # 故障节点
        ax1.add_patch(plt.Circle((x, y), 0.5, facecolor='red', edgecolor='darkred', linewidth=2, zorder=3))
        ax1.text(x, y, 'X', ha='center', va='center', fontsize=20, fontweight='bold', color='white', zorder=4)
        ax1.text(x, y-1, '故障!', ha='center', va='center', fontsize=10, color='red', fontweight='bold')
    elif name in ['A', 'B']:
        c = '#3498DB' if name == 'A' else '#E74C3C'
        ax1.add_patch(plt.Circle((x, y), 0.5, facecolor=c, edgecolor='black', linewidth=2, zorder=3))
        ax1.text(x, y, name, ha='center', va='center', fontsize=12, fontweight='bold', color='white', zorder=4)
    else:
        ax1.add_patch(plt.Circle((x, y), 0.4, facecolor='lightyellow', edgecolor='black', linewidth=1.5, zorder=3))
        ax1.text(x, y, name, ha='center', va='center', fontsize=10, zorder=4)

ax1.text(5, 0.8, '专用路径中的节点故障\n→ 整个连接断开，必须重新建立!',
         ha='center', va='center', fontsize=11, color='#E74C3C', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#FADBD8', edgecolor='#E74C3C'))

# 右图：分组交换 — 数据包绕道
ax2.set_xlim(0, 10); ax2.set_ylim(0, 9); ax2.axis('off')
ax2.set_title('分组交换：节点N2故障 → 数据包自动绕道', fontsize=14, fontweight='bold', color='#27AE60')

# 画所有边
for n1, n2 in edges:
    x1, y1 = nodes[n1]; x2, y2 = nodes[n2]
    ax2.plot([x1, x2], [y1, y2], '-', color='#DDD', linewidth=2)

# 新路径（绕过N2）
new_path = [('A','N3'), ('N3','N5'), ('N5','B')]
for n1, n2 in new_path:
    x1, y1 = nodes[n1]; x2, y2 = nodes[n2]
    ax2.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#27AE60', lw=3))

# 画节点
for name, (x, y) in nodes.items():
    if name == 'N2':
        ax2.add_patch(plt.Circle((x, y), 0.5, facecolor='red', edgecolor='darkred', linewidth=2, zorder=3))
        ax2.text(x, y, 'X', ha='center', va='center', fontsize=20, fontweight='bold', color='white', zorder=4)
        ax2.text(x, y-1, '故障!', ha='center', va='center', fontsize=10, color='red', fontweight='bold')
    elif name in ['A', 'B']:
        c = '#3498DB' if name == 'A' else '#27AE60'
        ax2.add_patch(plt.Circle((x, y), 0.5, facecolor=c, edgecolor='black', linewidth=2, zorder=3))
        ax2.text(x, y, name, ha='center', va='center', fontsize=12, fontweight='bold', color='white', zorder=4)
    else:
        on_path = name in ['N3', 'N5']
        fc = '#D5F5E3' if on_path else 'lightyellow'
        ec = '#27AE60' if on_path else 'black'
        lw = 2.5 if on_path else 1.5
        ax2.add_patch(plt.Circle((x, y), 0.4, facecolor=fc, edgecolor=ec, linewidth=lw, zorder=3))
        ax2.text(x, y, name, ha='center', va='center', fontsize=10, zorder=4)

ax2.text(5, 0.8, '路由器检测到N2故障\n→ 自动选择其他路径，通信继续!',
         ha='center', va='center', fontsize=11, color='#27AE60', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='#D5F5E3', edgecolor='#27AE60'))

plt.tight_layout()
plt.savefig('02_fault_tolerance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# === 交互式演示：分组交换完整流程 ===
import random

class PacketSwitchingFullDemo:
    '''完整演示分组交换过程：分割→路由→重组'''
    
    def __init__(self, data, packet_size=10):
        self.data = data
        self.packet_size = packet_size
        self.packets = []
        self.received = []
    
    def split_data(self):
        '''第一步：将数据分成多个包'''
        print('=== 第一步：数据分割 (Segmentation) ===')
        print(f'原始数据: "{self.data}"')
        print(f'数据包大小: {self.packet_size} 字符/包')
        print()
        
        for i in range(0, len(self.data), self.packet_size):
            chunk = self.data[i:i+self.packet_size]
            packet = {
                'seq': len(self.packets) + 1,
                'total': -1,  # 稍后填入
                'src': '192.168.1.100',
                'dst': '10.0.0.50',
                'ttl': 64,
                'data': chunk,
            }
            self.packets.append(packet)
        
        # 填入总包数
        for p in self.packets:
            p['total'] = len(self.packets)
        
        for p in self.packets:
            print(f'  Packet #{p["seq"]}/{p["total"]}: "{p["data"]}"')
        print(f'\n共 {len(self.packets)} 个数据包')
        print()
    
    def route_packets(self):
        '''第二步：每个包独立路由'''
        print('=== 第二步：路由传输 (Routing) ===')
        routes = [
            ['R1→R3→R5→R7'],
            ['R1→R2→R5→R7'],
            ['R1→R4→R6→R7'],
            ['R1→R2→R4→R6→R7'],
            ['R1→R3→R6→R7'],
        ]
        
        delays = []
        random.seed(42)
        for p in self.packets:
            route = random.choice(routes)[0]
            delay = random.uniform(10, 100)
            delays.append((delay, p))
            hops = route.count('→')
            print(f'  Packet #{p["seq"]}: 路径={route}, 跳数={hops}, 延迟={delay:.1f}ms')
        
        # 按延迟排序（模拟乱序到达）
        delays.sort(key=lambda x: x[0])
        self.received = [p for _, p in delays]
        print()
        print('  到达顺序:', [p['seq'] for p in self.received])
        print('  注意：数据包到达顺序与发送顺序不同！')
        print()
    
    def reassemble(self):
        '''第三步：按序列号重组'''
        print('=== 第三步：重组 (Reassembly) ===')
        print('收到的数据包（乱序）:')
        for p in self.received:
            print(f'  Packet #{p["seq"]}: "{p["data"]}"')
        print()
        
        # 按序列号排序
        sorted_packets = sorted(self.received, key=lambda x: x['seq'])
        reassembled = ''.join(p['data'] for p in sorted_packets)
        
        print('按序列号排序后:')
        for p in sorted_packets:
            print(f'  Packet #{p["seq"]}: "{p["data"]}"')
        print()
        print(f'重组结果: "{reassembled}"')
        print(f'与原始数据一致: {reassembled == self.data}')
    
    def run(self):
        print('=' * 60)
        print('分组交换完整流程演示')
        print('=' * 60)
        print()
        self.split_data()
        self.route_packets()
        self.reassemble()
        print()
        print('=' * 60)

demo = PacketSwitchingFullDemo(
    'Cambridge A-Level Computer Science is fascinating!',
    packet_size=12
)


In [ ]:
# === 可视化：数据包重组过程 ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 10))
fig.suptitle('数据包的分割、乱序到达、重组过程', fontsize=18, fontweight='bold')

# 模拟数据
original = 'HELLO WORLD FROM CHINA!'
pkt_size = 6
packets = [original[i:i+pkt_size] for i in range(0, len(original), pkt_size)]
n = len(packets)
colors = plt.cm.Set3(np.linspace(0, 1, n))

# 上图：发送顺序
ax1.set_xlim(-0.5, n+0.5); ax1.set_ylim(-0.5, 1.5); ax1.axis('off')
ax1.set_title('第一步：发送方按顺序发出', fontsize=13, color='#2C3E50')
for i, (pkt, color) in enumerate(zip(packets, colors)):
    ax1.add_patch(mpatches.FancyBboxPatch((i, 0), 0.9, 1, boxstyle='round',
                                           facecolor=color, edgecolor='black', linewidth=2))
    ax1.text(i+0.45, 0.7, f'#{i+1}', ha='center', va='center', fontsize=10, fontweight='bold')
    ax1.text(i+0.45, 0.3, f'"{pkt}"', ha='center', va='center', fontsize=8)
ax1.annotate('发送方', xy=(-0.3, 0.5), fontsize=11, fontweight='bold', color='#3498DB')

# 中图：乱序到达
arrival_order = [2, 0, 3, 1]  # 到达顺序
if n > 4:
    arrival_order = list(range(n))
    import random; random.seed(42); random.shuffle(arrival_order)

ax2.set_xlim(-0.5, n+0.5); ax2.set_ylim(-0.5, 1.5); ax2.axis('off')
ax2.set_title('第二步：乱序到达（走了不同路径，延迟不同）', fontsize=13, color='#E74C3C')
for pos, orig_idx in enumerate(arrival_order):
    ax2.add_patch(mpatches.FancyBboxPatch((pos, 0), 0.9, 1, boxstyle='round',
                                           facecolor=colors[orig_idx], edgecolor='black', linewidth=2))
    ax2.text(pos+0.45, 0.7, f'#{orig_idx+1}', ha='center', va='center', fontsize=10, fontweight='bold')
    ax2.text(pos+0.45, 0.3, f'"{packets[orig_idx]}"', ha='center', va='center', fontsize=8)
ax2.annotate('接收方', xy=(-0.3, 0.5), fontsize=11, fontweight='bold', color='#E74C3C')

# 下图：重组
ax3.set_xlim(-0.5, n+0.5); ax3.set_ylim(-0.5, 1.5); ax3.axis('off')
ax3.set_title('第三步：按序列号重组 → 恢复原始数据', fontsize=13, color='#27AE60')
for i, (pkt, color) in enumerate(zip(packets, colors)):
    ax3.add_patch(mpatches.FancyBboxPatch((i, 0), 0.9, 1, boxstyle='round',
                                           facecolor=color, edgecolor='#27AE60', linewidth=3))
    ax3.text(i+0.45, 0.7, f'#{i+1}', ha='center', va='center', fontsize=10, fontweight='bold')
    ax3.text(i+0.45, 0.3, f'"{pkt}"', ha='center', va='center', fontsize=8)
ax3.text(n/2, -0.3, f'重组结果: "{original}"', ha='center', va='center', fontsize=12,
         fontweight='bold', color='#27AE60')

plt.tight_layout()
plt.savefig('02_reassembly.png', dpi=150, bbox_inches='tight')


---
## 6. 实际应用对比

| 应用场景 | 使用的交换方式 | 为什么 |
|----------|---------------|--------|
| 传统电话 (PSTN) | 电路交换 | 需要恒定延迟，保证语音质量 |
| 网页浏览 | 分组交换 | 数据突发性强，需要灵活路由 |
| 文件下载 | 分组交换 | 大文件可分块传输，丢包可重传 |
| 视频通话 (VoIP) | 分组交换+QoS | 现代互联网通过QoS优先级保证质量 |
| 银行交易 | 分组交换+加密 | 需要可靠性，但不需要实时性 |

**有趣的事实**：现代电话（VoIP，如微信语音）已经不再用电路交换了！它们使用分组交换 + 服务质量(QoS)保证来实现实时通话。这说明分组交换已经强大到可以应对几乎所有场景。

In [ ]:
# === 总结可视化：电路交换 vs 分组交换 决策流程 ===
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14); ax.set_ylim(0, 12); ax.axis('off')
ax.set_title('总结：选择哪种交换方式？', fontsize=18, fontweight='bold', pad=20)

# 对比表格
headers = ['特征', '电路交换', '分组交换']
rows = [
    ['连接', '需要预先建立', '无需建立'],
    ['路径', '固定不变', '动态选择'],
    ['资源', '独占', '共享'],
    ['效率', '低(有浪费)', '高(充分利用)'],
    ['容错', '差(单点故障)', '强(自动绕道)'],
    ['顺序', '保证顺序', '可能乱序'],
    ['延迟', '恒定', '可能波动'],
    ['应用', '传统电话', '互联网'],
]

# 绘制表格
col_widths = [3, 4, 4]
col_starts = [1.5, 4.5, 8.5]
row_height = 0.9
start_y = 10.5

# 表头
for j, (header, x, w) in enumerate(zip(headers, col_starts, col_widths)):
    color = '#2C3E50' if j == 0 else ('#E74C3C' if j == 1 else '#27AE60')
    ax.add_patch(mpatches.FancyBboxPatch((x, start_y), w, row_height, boxstyle='round,pad=0.05',
                                          facecolor=color, edgecolor='black', linewidth=1.5))
    ax.text(x + w/2, start_y + row_height/2, header, ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')

# 数据行
for i, row in enumerate(rows):
    y = start_y - (i + 1) * row_height - 0.1
    for j, (cell, x, w) in enumerate(zip(row, col_starts, col_widths)):
        if j == 0:
            fc = '#F8F9FA'
        elif j == 1:
            fc = '#FADBD8' if i % 2 == 0 else '#F9EBEA'
        else:
            fc = '#D5F5E3' if i % 2 == 0 else '#EAFAF1'
        ax.add_patch(mpatches.FancyBboxPatch((x, y), w, row_height, boxstyle='round,pad=0.05',
                                              facecolor=fc, edgecolor='#BDC3C7', linewidth=1))
        ax.text(x + w/2, y + row_height/2, cell, ha='center', va='center',
                fontsize=10, fontweight='bold' if j == 0 else 'normal')

# 底部总结
ax.text(7, 1.5, '考试答题策略：题目问"比较两种交换方式"时，\n从上面的表格中选4-5个特征进行对比。\n每个特征都要说明两种方式的不同之处。',
        ha='center', va='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#FEF9E7', edgecolor='#F39C12', linewidth=2))

plt.tight_layout()
plt.savefig('02_summary_table.png', dpi=150, bbox_inches='tight')


---
## 7. 练习题 (Exercises)

### 练习 1：简答题

1. 解释电路交换和分组交换的区别。(4分)

2. 描述一个数据包(packet)的报头中包含哪些关键信息，并解释每个字段的作用。(6分)

3. 解释路由器在分组交换网络中的作用。(3分)

4. 为什么在分组交换中，数据包可能乱序到达？接收方如何解决这个问题？(3分)

5. 解释为什么互联网使用分组交换而不是电路交换。给出至少三个原因。(6分)

6. TTL(Time to Live)字段的作用是什么？如果没有这个字段会发生什么？(2分)

### 练习 2：场景分析题

一家公司需要在两个办公室之间传输大量数据文件。请分析使用电路交换和分组交换各有什么优缺点，并推荐一种方式。

In [ ]:
# === 练习：判断题自测 ===

def switching_quiz():
    '''电路交换 vs 分组交换 判断题'''
    questions = [
        ('电路交换在通信前需要建立专用连接', True, '电路交换的第一步是建立连接(Circuit Establishment)'),
        ('分组交换中，所有数据包走同一条路径', False, '每个数据包独立路由，可能走不同路径'),
        ('电路交换的带宽利用率比分组交换高', False, '电路交换独占带宽，空闲时浪费资源'),
        ('分组交换中，数据包一定按顺序到达', False, '数据包可能乱序到达，需按序列号重组'),
        ('路由器的作用是根据路由表转发数据包', True, '路由器检查目标IP，查路由表，决定下一跳'),
        ('电路交换更适合互联网的突发性流量', False, '分组交换更适合突发性流量，能共享带宽'),
        ('TTL防止数据包在网络中无限循环', True, 'TTL每经过一个路由器减1，到0就丢弃'),
        ('传统电话网络(PSTN)使用分组交换', False, '传统电话网络使用电路交换'),
    ]
    
    print('=== 电路交换 vs 分组交换 判断题 ===')
    print('请判断以下说法是否正确 (T/F)\n')
    
    score = 0
    for i, (question, correct, explanation) in enumerate(questions, 1):
        try:
            answer = input(f'{i}. {question}\n   你的答案 (T/F): ')
            user_answer = answer.strip().upper() in ['T', 'TRUE', '对', '正确']
            if user_answer == correct:
                print(f'   正确!')
                score += 1
            else:
                print(f'   错误!')
            print(f'   解释: {explanation}\n')
        except EOFError:
            result = '正确(T)' if correct else '错误(F)'
            print(f'   答案: {result}')
            print(f'   解释: {explanation}\n')
            score += 1
    
    print(f'得分: {score}/{len(questions)}')
    if score >= 7:
        print('太棒了！你已经很好地掌握了这部分内容！')
    elif score >= 5:
        print('不错！再复习一下薄弱的知识点。')
    else:
        print('需要再仔细看看课件内容哦！')



---
## 本节总结 (Summary)

### 核心考点回顾

| 考点 | 关键内容 |
|------|----------|
| 电路交换 | 专用通路、三个阶段、资源独占、固定延迟 |
| 分组交换 | 数据分包、独立路由、共享带宽、可能乱序 |
| 数据包结构 | Header(源IP/目标IP/序列号/TTL) + Payload + Trailer(Checksum) |
| 路由器 | 查路由表、转发数据包、TTL减1 |
| 互联网选择分组交换的原因 | 突发性流量、容错性、可扩展性、成本低 |

### 考试答题模板

当题目要求"比较电路交换和分组交换"时，你的答案应该包含：
- 电路交换需要建立专用连接，分组交换不需要
- 电路交换路径固定，分组交换每个包可能走不同路径
- 电路交换独占资源（效率低），分组交换共享资源（效率高）
- 电路交换数据按序到达，分组交换可能乱序需重组
- 电路交换容错性差，分组交换可自动绕过故障

### 全章回顾
至此，第14章的两个核心主题已经学完：
1. **通信协议与TCP/IP** — 网络通信的"规则"
2. **电路交换与分组交换** — 数据传输的两种"方式"